# Deep Research Agent with LangChain DeepAgents
Based on the official research_agent.py example from langchain-ai/deepagents

In [1]:
# =================================================================
# CELL 1: Installation and Setup
# =================================================================

# Import necessary libraries
import os
import asyncio
from typing import Literal, Dict, Any, List
from dotenv import load_dotenv

from tavily import TavilyClient
from deepagents import create_deep_agent

# Load environment variables
load_dotenv()

True

In [2]:
# =================================================================
# CELL 2: Environment Configuration
# =================================================================

# Set up API keys (you'll need to add these to your environment)
# Create a .env file with your API keys or set them directly

# OpenAI API Key for the LLM
openai_api_key = os.getenv("OPENAI_API_KEY") or input("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key

# Tavily API Key for web search (optional - you can use other search tools)
tavily_api_key = os.getenv("TAVILY_API_KEY") or input("Enter your Tavily API key (or press Enter to skip): ")
if tavily_api_key:
    os.environ["TAVILY_API_KEY"] = tavily_api_key

print("✅ Environment setup complete!")

✅ Environment setup complete!


In [3]:
# =================================================================
# CELL 3: Define Research Tools
# =================================================================

# Initialize Tavily client for web search
if tavily_api_key:
    tavily_client = TavilyClient(api_key=tavily_api_key)

def internet_search(query: str) -> str:
    """
    Search the web for information using Tavily API.
    
    Args:
        query: The search query string
        
    Returns:
        Search results as formatted text
    """
    if not tavily_api_key:
        return "Web search unavailable - no Tavily API key provided"
    
    try:
        response = tavily_client.search(
            query=query,
            search_depth="advanced",
            max_results=5
        )
        
        results = []
        for result in response.get('results', []):
            results.append(f"**{result['title']}**\n{result['content']}\nSource: {result['url']}\n")
        
        return "\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"

In [4]:
# =================================================================
# CELL 4: Define Sub-Agents (from official example)
# =================================================================

# Research Sub-Agent Configuration
sub_research_prompt = """You are a dedicated researcher. Your job is to conduct research based on the users questions.

Conduct thorough research and then reply to the user with a detailed answer to their question

only your FINAL answer will be passed on to the user. They will have NO knowledge of anything except your final message, so your final report should be your final message!"""

research_sub_agent = {
    "name": "research-agent",
    "description": "Used to research more in depth questions. Only give this researcher one topic at a time. Do not pass multiple sub questions to this researcher. Instead, you should break down a large topic into the necessary components, and then call multiple research agents in parallel, one for each sub question.",
    "prompt": sub_research_prompt,
    "tools": ["internet_search"],
}

# Critique Sub-Agent Configuration
sub_critique_prompt = """You are a dedicated editor. You are being tasked to critique a report.

You can find the report at `final_report.md`.

You can find the question/topic for this report at `question.txt`.

The user may ask for specific areas to critique the report in. Respond to the user with a detailed critique of the report. Things that could be improved.

You can use the search tool to search for information, if that will help you critique the report

Do not write to the `final_report.md` yourself.

Things to check:
- Check that each section is appropriately named
- Check that the report is written as you would find in an essay or a textbook - it should be text heavy, do not let it just be a list of bullet points!
- Check that the report is comprehensive. If any paragraphs or sections are short, or missing important details, point it out.
- Check that the article covers key areas of the industry, ensures overall understanding, and does not omit important parts.
- Check that the article deeply analyzes causes, impacts, and trends, providing valuable insights
- Check that the article closely follows the research topic and directly answers questions
- Check that the article has a clear structure, fluent language, and is easy to understand.
"""

critique_sub_agent = {
    "name": "critique-agent", 
    "description": "Used to critique the final report. Give this agent some information about how you want it to critique the report.",
    "prompt": sub_critique_prompt,
}

print("🤖 Sub-agents configured successfully!")


🤖 Sub-agents configured successfully!


In [5]:
# =================================================================
# CELL 5: Main Research Agent Instructions (from official example)
# =================================================================

research_instructions = """You are an expert researcher. Your job is to conduct thorough research, and then write a polished report.

The first thing you should do is to write the original user question to `question.txt` so you have a record of it.

Use the research-agent to conduct deep research. It will respond to your questions/topics with a detailed answer.

When you think you enough information to write a final report, write it to `final_report.md`

You can call the critique-agent to get a critique of the final report. After that (if needed) you can do more research and edit the `final_report.md`
You can do this however many times you want until are you satisfied with the result.

Only edit the file once at a time (if you call this tool in parallel, there may be conflicts).

Here are instructions for writing the final report:

<report_instructions>

CRITICAL: Make sure the answer is written in the same language as the human messages! If you make a todo plan - you should note in the plan what language the report should be in so you dont forget!
Note: the language the report should be in is the language the QUESTION is in, not the language/country that the question is ABOUT.

Please create a detailed answer to the overall research brief that:
1. Is well-organized with proper headings (# for title, ## for sections, ### for subsections)
2. Includes specific facts and insights from the research
3. References relevant sources using [Title](URL) format
4. Provides a balanced, thorough analysis. Be as comprehensive as possible, and include all information that is relevant to the overall research question. People are using you for deep research and will expect detailed, comprehensive answers.
5. Includes a "Sources" section at the end with all referenced links

You can structure your report in a number of different ways. Here are some examples:

To answer a question that asks you to compare two things, you might structure your report like this:
1/ intro
2/ overview of topic A
3/ overview of topic B
4/ comparison between A and B
5/ conclusion

To answer a question that asks you to return a list of things, you might only need a single section which is the entire list.
1/ list of things or table of things
Or, you could choose to make each item in the list a separate section in the report. When asked for lists, you don't need an introduction or conclusion.
1/ item 1
2/ item 2
3/ item 3

To answer a question that asks you to summarize a topic, give a report, or give an overview, you might structure your report like this:
1/ overview of topic
2/ concept 1
3/ concept 2
4/ concept 3
5/ conclusion

If you think you can answer the question with a single section, you can do that too!
1/ answer

REMEMBER: Section is a VERY fluid and loose concept. You can structure your report however you think is best, including in ways that are not listed above!
Make sure that your sections are cohesive, and make sense for the reader.

For each section of the report, do the following:
- Use simple, clear language
- Use ## for section title (Markdown format) for each section of the report
- Do NOT ever refer to yourself as the writer of the report. This should be a professional report without any self-referential language. 
- Do not say what you are doing in the report. Just write the report without any commentary from yourself.
- Each section should be as long as necessary to deeply answer the question with the information you have gathered. It is expected that sections will be fairly long and verbose. You are writing a deep research report, and users will expect a thorough answer.
- Use bullet points to list out information when appropriate, but by default, write in paragraph form.

REMEMBER:
The brief and research may be in English, but you need to translate this information to the right language when writing the final answer.
Make sure the final answer report is in the SAME language as the human messages in the message history.

Format the report in clear markdown with proper structure and include source references where appropriate.

<Citation Rules>
- Assign each unique URL a single citation number in your text
- End with ### Sources that lists each source with corresponding numbers
- IMPORTANT: Number sources sequentially without gaps (1,2,3,4...) in the final list regardless of which sources you choose
- Each source should be a separate line item in a list, so that in markdown it is rendered as a list.
- Example format:
  [1] Source Title: URL
  [2] Source Title: URL
- Citations are extremely important. Make sure to include these, and pay a lot of attention to getting these right. Users will often use these citations to look into more information.
</Citation Rules>
</report_instructions>

You have access to a few tools.

## `internet_search`

Use this to run an internet search for a given query. You can specify the number of results, the topic, and whether raw content should be included.
"""

print("📋 Research instructions loaded!")

📋 Research instructions loaded!


In [6]:
# =================================================================
# CELL 6: Create the Research Agent
# =================================================================

# Create the main research agent with all components
agent = create_deep_agent(
    tools=[internet_search],
    instructions=research_instructions,
    subagents=[critique_sub_agent, research_sub_agent],
).with_config({"recursion_limit": 1000})

print("🎉 Deep research agent created successfully!")
print("Features:")
print("- ✅ Web search capabilities via Tavily")
print("- ✅ Dedicated research sub-agent")
print("- ✅ Built-in critique and editing system")
print("- ✅ File system for saving reports")
print("- ✅ Citation management")

🎉 Deep research agent created successfully!
Features:
- ✅ Web search capabilities via Tavily
- ✅ Dedicated research sub-agent
- ✅ Built-in critique and editing system
- ✅ File system for saving reports
- ✅ Citation management


In [7]:
# =================================================================
# CELL 7: Helper Functions for Testing
# =================================================================

def format_agent_response(response):
    """Format and display the agent response nicely."""
    print("\n" + "="*80)
    print("🤖 RESEARCH AGENT RESPONSE")
    print("="*80)
    
    try:
        # Handle different response formats from deepagents
        if hasattr(response, 'content'):
            # If it's an AIMessage object
            print(response.content)
        elif isinstance(response, dict):
            if 'messages' in response:
                # If it's a dict with messages
                final_message = response['messages'][-1]
                if hasattr(final_message, 'content'):
                    print(final_message.content)
                elif isinstance(final_message, dict) and 'content' in final_message:
                    print(final_message['content'])
                else:
                    print(final_message)
            else:
                # If it's just a dict
                print(response)
        else:
            # Fallback - just print whatever it is
            print(response)
    except Exception as e:
        print(f"Error formatting response: {e}")
        print(f"Response type: {type(response)}")
        print(f"Response: {response}")
    
    print("="*80)

def run_research(query: str, verbose: bool = True):
    """
    Run a research query through the agent.
    
    Args:
        query: The research question or topic
        verbose: Whether to print detailed output
        
    Returns:
        The agent's response
    """
    if verbose:
        print(f"🔍 Research Query: {query}")
        print("⏳ Starting research process...")
        print("Note: This may take several minutes for complex queries...")
    
    try:
        # Run the agent - try different input formats
        response = agent.invoke({
            "messages": [{"role": "user", "content": query}]
        })
        
        if verbose:
            format_agent_response(response)
        
        return response
        
    except Exception as e:
        error_msg = f"❌ Error during research: {str(e)}"
        print(error_msg)
        
        # Let's also try a simpler input format
        try:
            if verbose:
                print("🔄 Trying alternative input format...")
            response = agent.invoke(query)
            if verbose:
                format_agent_response(response)
            return response
        except Exception as e2:
            error_msg2 = f"❌ Alternative format also failed: {str(e2)}"
            print(error_msg2)
            return {"error": error_msg, "alternative_error": error_msg2}

def run_research_streaming(query: str):
    """
    Run research with streaming output to see progress in real-time.
    
    Args:
        query: The research question or topic
    """
    print(f"🔍 Research Query: {query}")
    print("⏳ Streaming research process...")
    print("-" * 80)
    
    try:
        for chunk in agent.stream({
            "messages": [{"role": "user", "content": query}]
        }):
            # Print intermediate results as they come
            if 'agent' in chunk:
                latest_message = chunk['agent']['messages'][-1]
                if latest_message['role'] == 'assistant':
                    print(f"🤖 Agent: {latest_message['content'][:200]}...")
            elif 'research-agent' in chunk:
                print(f"🔬 Research Sub-agent working...")
            elif 'critique-agent' in chunk:
                print(f"✏️ Critique Sub-agent reviewing...")
                
    except Exception as e:
        print(f"❌ Streaming error: {str(e)}")


In [8]:
# =================================================================
# CELL 8: Test Cases and Examples
# =================================================================

# Define comprehensive test queries
test_queries = {
    "quick": [
        "What are the key benefits of electric vehicles?",
        "Explain quantum computing in simple terms",
        "What is the current state of renewable energy adoption?"
    ],
    "medium": [
        "Compare the economic impacts of remote work vs. in-office work",
        "Analyze the latest developments in AI regulation across different countries",
        "What are the environmental and economic trade-offs of nuclear energy?"
    ],
    "complex": [
        "Provide a comprehensive analysis of the global semiconductor shortage: causes, impacts, and long-term solutions",
        "Research the effectiveness of different carbon capture technologies and their potential for large-scale deployment",
        "Analyze the geopolitical implications of the transition to renewable energy sources"
    ]
}

print("📋 Available test queries by complexity:")
print("\n🟢 QUICK (2-3 minutes):")
for i, query in enumerate(test_queries["quick"], 1):
    print(f"  {i}. {query}")

print("\n🟡 MEDIUM (5-7 minutes):")
for i, query in enumerate(test_queries["medium"], 1):
    print(f"  {i}. {query}")

print("\n🔴 COMPLEX (10-15 minutes):")
for i, query in enumerate(test_queries["complex"], 1):
    print(f"  {i}. {query}")

📋 Available test queries by complexity:

🟢 QUICK (2-3 minutes):
  1. What are the key benefits of electric vehicles?
  2. Explain quantum computing in simple terms
  3. What is the current state of renewable energy adoption?

🟡 MEDIUM (5-7 minutes):
  1. Compare the economic impacts of remote work vs. in-office work
  2. Analyze the latest developments in AI regulation across different countries
  3. What are the environmental and economic trade-offs of nuclear energy?

🔴 COMPLEX (10-15 minutes):
  1. Provide a comprehensive analysis of the global semiconductor shortage: causes, impacts, and long-term solutions
  2. Research the effectiveness of different carbon capture technologies and their potential for large-scale deployment
  3. Analyze the geopolitical implications of the transition to renewable energy sources


In [9]:
# =================================================================
# CELL 9: Quick Test Function
# =================================================================

def quick_test():
    """Run a quick test to verify the agent is working."""
    test_query = "I do a lot AI engineering on my laptop. What are the best laptops under 2k for creating AI apps?"
    print("🚀 Running quick test...")
    return run_research(test_query, verbose=True)

# Uncomment the line below to run a quick test
quick_test()

🚀 Running quick test...
🔍 Research Query: I do a lot AI engineering on my laptop. What are the best laptops under 2k for creating AI apps?
⏳ Starting research process...
Note: This may take several minutes for complex queries...

🤖 RESEARCH AGENT RESPONSE
I've completed comprehensive research on the best laptops under $2,000 for AI engineering and app development. The final report covers all the essential aspects you need to make an informed decision.

## Key Findings Summary:

**Top Recommendations:**
1. **HP Omen with RTX 4070** ($1,400-$1,600) - Best overall value for serious AI work
2. **Lenovo Legion Pro 5** ($1,800-$2,000) - Maximum performance with 32GB RAM
3. **ASUS TUF Gaming A15** ($1,200-$1,400) - Best entry-level option for learning

**Critical Specifications for AI Work:**
- **GPU:** RTX 4060 (8GB VRAM) minimum, RTX 4070 (12GB VRAM) preferred
- **RAM:** 16GB minimum, 32GB strongly recommended for professional work
- **CPU:** Modern multi-core processors (AMD Ryzen 7/9 or I

{'messages': [HumanMessage(content='I do a lot AI engineering on my laptop. What are the best laptops under 2k for creating AI apps?', additional_kwargs={}, response_metadata={}, id='461e0df7-6abb-4335-b5bb-4f6dabfea9ab'),
  AIMessage(content=[{'text': "I'll help you research the best laptops under $2,000 for AI engineering and creating AI apps. Let me start by documenting your question and creating a research plan.", 'type': 'text'}, {'id': 'toolu_01Y3t36zU8yLG6EFMevtYjSS', 'input': {'file_path': 'question.txt', 'content': 'I do a lot AI engineering on my laptop. What are the best laptops under 2k for creating AI apps?'}, 'name': 'write_file', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_015LY2gJWHvbt2cbtYyLTuJ1', 'model': 'claude-sonnet-4-20250514', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, '

In [27]:
# =================================================================
# CELL 10: Interactive Research Session
# =================================================================

def interactive_research_session():
    """Start an interactive research session."""
    print("\n🚀 Starting Interactive Deep Research Session")
    print("Commands:")
    print("  - Type your research question to start research")
    print("  - Type 'test quick/medium/complex' to run predefined tests")
    print("  - Type 'stream <question>' to see streaming output")
    print("  - Type 'quit' to exit")
    print("-" * 80)
    
    while True:
        try:
            user_input = input("\n🔍 Enter command or research question: ").strip()
            
            if user_input.lower() == 'quit':
                print("👋 Goodbye!")
                break
            elif user_input.lower().startswith('test '):
                complexity = user_input.split(' ', 1)[1].lower()
                if complexity in test_queries:
                    query = test_queries[complexity][0]  # Use first query
                    print(f"Running {complexity} test query...")
                    run_research(query)
                else:
                    print("Available tests: quick, medium, complex")
            elif user_input.lower().startswith('stream '):
                query = user_input[7:]  # Remove 'stream ' prefix
                run_research_streaming(query)
            elif user_input:
                run_research(user_input)
            else:
                print("Please enter a valid command or research question.")
                
        except KeyboardInterrupt:
            print("\n👋 Session interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error: {str(e)}")

interactive_research_session()


🚀 Starting Interactive Deep Research Session
Commands:
  - Type your research question to start research
  - Type 'test quick/medium/complex' to run predefined tests
  - Type 'stream <question>' to see streaming output
  - Type 'quit' to exit
--------------------------------------------------------------------------------



🔍 Enter command or research question:  compare laptops across speed and specs. I'm considering buying a Macbook Air. I currently have a Macbook Pro. I conduct Gen AI research, so I build a lot of Agentic apps locally. What laptop is most suitable for my needs?


🔍 Research Query: compare laptops across speed and specs. I'm considering buying a Macbook Air. I currently have a Macbook Pro. I conduct Gen AI research, so I build a lot of Agentic apps locally. What laptop is most suitable for my needs?
⏳ Starting research process...
Note: This may take several minutes for complex queries...

🤖 RESEARCH AGENT RESPONSE
## Summary

I've completed a comprehensive analysis of laptop options for your Gen AI research and agentic app development needs. Here are the key findings:

**Main Recommendation:** Don't downgrade from your MacBook Pro to a MacBook Air. For your use case, I recommend upgrading to a **MacBook Pro M3 Pro with 36GB RAM**, which offers the best balance of performance, thermal management, and cost-effectiveness for sustained AI workloads.

**Key Insights:**
- MacBook Air's fanless design will throttle performance during sustained AI training, reducing productivity
- Memory is critical for agentic apps - 24GB+ is recommended, with 36GB+ be


🔍 Enter command or research question:  quit


👋 Goodbye!
